In [1]:
!pip install clickhouse-connect pandas numpy

# CODE GENERATE

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import clickhouse_connect

# --- Danh sách ngày lễ Việt Nam cơ bản (ví dụ) ---
HOLIDAYS = [
    "2026-01-01",
    "2025-01-01", "2025-02-10", "2025-04-30", "2025-05-01", "2025-09-02",
    "2024-01-01", "2024-02-10", "2024-04-30", "2024-05-01", "2024-09-02",
    "2023-01-01", "2023-02-10", "2023-04-30", "2023-05-01", "2023-09-02",
    "2022-01-01", "2022-02-10", "2022-04-30", "2022-05-01", "2022-09-02",
    "2021-01-01", "2021-02-10", "2021-04-30", "2021-05-01", "2021-09-02",
    "2020-01-01", "2020-02-10", "2020-04-30", "2020-05-01", "2020-09-02",
]

def is_holiday(dt):
    return dt.strftime("%Y-%m-%d") in HOLIDAYS or dt.weekday() >= 5  # cuối tuần coi là ô nhiễm cao

# Danh sách sensor mới cho 34 tỉnh/thành Việt Nam
additional_sensors = {
    "3276373": {"area": "Cao Bằng", "location_name": "Việt Nam", "latitude": 22.6666, "longitude": 106.2639, "owner_name": "Hoaze", "provider": "ProviderCB", "unit": "µg/m3"},
    "3276374": {"area": "Sơn La", "location_name": "Việt Nam", "latitude": 21.3256, "longitude": 103.9188, "owner_name": "Hoaze", "provider": "ProviderSL", "unit": "µg/m3"},
    "3276375": {"area": "Lai Châu", "location_name": "Việt Nam", "latitude": 22.3964, "longitude": 103.4582, "owner_name": "Hoaze", "provider": "ProviderLC", "unit": "µg/m3"},
    "3276376": {"area": "Lạng Sơn", "location_name": "Việt Nam", "latitude": 21.8537, "longitude": 106.7615, "owner_name": "Hoaze", "provider": "ProviderLS", "unit": "µg/m3"},
    "3276377": {"area": "Tuyên Quang", "location_name": "Việt Nam", "latitude": 21.8236, "longitude": 105.2181, "owner_name": "Hoaze", "provider": "ProviderTQ", "unit": "µg/m3"},
    "3276378": {"area": "Lào Cai", "location_name": "Việt Nam", "latitude": 22.4856, "longitude": 103.9707, "owner_name": "Hoaze", "provider": "ProviderLC2", "unit": "µg/m3"},
    "3276379": {"area": "Thái Nguyên", "location_name": "Việt Nam", "latitude": 21.5942, "longitude": 105.8480, "owner_name": "Hoaze", "provider": "ProviderTN", "unit": "µg/m3"},
    "3276380": {"area": "Điện Biên", "location_name": "Việt Nam", "latitude": 21.3860, "longitude": 103.0230, "owner_name": "Hoaze", "provider": "ProviderDB", "unit": "µg/m3"},
    "3276381": {"area": "Phú Thọ", "location_name": "Việt Nam", "latitude": 21.3227, "longitude": 105.4016, "owner_name": "Hoaze", "provider": "ProviderPT", "unit": "µg/m3"},
    "3276382": {"area": "Bắc Ninh", "location_name": "Việt Nam", "latitude": 21.1861, "longitude": 106.0763, "owner_name": "Hoaze", "provider": "ProviderBN", "unit": "µg/m3"},
    "3276383": {"area": "Hà Nội", "location_name": "Việt Nam", "latitude": 21.0278, "longitude": 105.8342, "owner_name": "Hoaze", "provider": "ProviderHN", "unit": "µg/m3"},
    "3276384": {"area": "Quảng Ninh", "location_name": "Việt Nam", "latitude": 21.0064, "longitude": 107.2925, "owner_name": "Hoaze", "provider": "ProviderQN", "unit": "µg/m3"},
    "3276385": {"area": "Hải Phòng", "location_name": "Việt Nam", "latitude": 20.8449, "longitude": 106.6881, "owner_name": "Hoaze", "provider": "ProviderHP", "unit": "µg/m3"},
    "3276386": {"area": "Hưng Yên", "location_name": "Việt Nam", "latitude": 20.6464, "longitude": 106.0511, "owner_name": "Hoaze", "provider": "ProviderHY", "unit": "µg/m3"},
    "3276387": {"area": "Ninh Bình", "location_name": "Việt Nam", "latitude": 20.2506, "longitude": 105.9745, "owner_name": "Hoaze", "provider": "ProviderNB", "unit": "µg/m3"},
    "3276388": {"area": "Thanh Hóa", "location_name": "Việt Nam", "latitude": 19.8067, "longitude": 105.7852, "owner_name": "Hoaze", "provider": "ProviderTH", "unit": "µg/m3"},
    "3276389": {"area": "Nghệ An", "location_name": "Việt Nam", "latitude": 18.6734, "longitude": 105.6923, "owner_name": "Hoaze", "provider": "ProviderNA", "unit": "µg/m3"},
    "3276390": {"area": "Hà Tĩnh", "location_name": "Việt Nam", "latitude": 18.3428, "longitude": 105.9057, "owner_name": "Hoaze", "provider": "ProviderHT", "unit": "µg/m3"},
    "3276391": {"area": "Quảng Bình", "location_name": "Việt Nam", "latitude": 17.4689, "longitude": 106.6223, "owner_name": "Hoaze", "provider": "ProviderQB", "unit": "µg/m3"},
    "3276392": {"area": "Thừa Thiên Huế", "location_name": "Việt Nam", "latitude": 16.4637, "longitude": 107.5909, "owner_name": "Hoaze", "provider": "ProviderTTH", "unit": "µg/m3"},
    "3276393": {"area": "Đà Nẵng", "location_name": "Việt Nam", "latitude": 16.0544, "longitude": 108.2022, "owner_name": "Hoaze", "provider": "ProviderDN", "unit": "µg/m3"},
    "3276394": {"area": "Quảng Ngãi", "location_name": "Việt Nam", "latitude": 15.1214, "longitude": 108.8044, "owner_name": "Hoaze", "provider": "ProviderQN2", "unit": "µg/m3"},
    "3276395": {"area": "Gia Lai", "location_name": "Việt Nam", "latitude": 13.9833, "longitude": 108.0000, "owner_name": "Hoaze", "provider": "ProviderGL", "unit": "µg/m3"},
    "3276396": {"area": "Đắk Lắk", "location_name": "Việt Nam", "latitude": 12.6667, "longitude": 108.0500, "owner_name": "Hoaze", "provider": "ProviderDL", "unit": "µg/m3"},
    "3276397": {"area": "Khánh Hòa", "location_name": "Việt Nam", "latitude": 12.2388, "longitude": 109.1967, "owner_name": "Hoaze", "provider": "ProviderKH", "unit": "µg/m3"},
    "3276398": {"area": "Lâm Đồng", "location_name": "Việt Nam", "latitude": 11.9404, "longitude": 108.4583, "owner_name": "Hoaze", "provider": "ProviderLD", "unit": "µg/m3"},
    "3276399": {"area": "Đồng Nai", "location_name": "Việt Nam", "latitude": 10.9574, "longitude": 106.8427, "owner_name": "Hoaze", "provider": "ProviderDN3", "unit": "µg/m3"},
    "3276400": {"area": "TP. Hồ Chí Minh", "location_name": "Việt Nam", "latitude": 10.7758, "longitude": 106.7019, "owner_name": "Hoaze", "provider": "ProviderHCM", "unit": "µg/m3"},
    "3276401": {"area": "Tây Ninh", "location_name": "Việt Nam", "latitude": 11.3100, "longitude": 106.0983, "owner_name": "Hoaze", "provider": "ProviderTN2", "unit": "µg/m3"},
    "3276402": {"area": "Đồng Tháp", "location_name": "Việt Nam", "latitude": 10.4590, "longitude": 105.6355, "owner_name": "Hoaze", "provider": "ProviderDT", "unit": "µg/m3"},
    "3276403": {"area": "Vĩnh Long", "location_name": "Việt Nam", "latitude": 10.2530, "longitude": 105.9722, "owner_name": "Hoaze", "provider": "ProviderVL", "unit": "µg/m3"},
    "3276404": {"area": "Cần Thơ", "location_name": "Việt Nam", "latitude": 10.0452, "longitude": 105.7469, "owner_name": "Hoaze", "provider": "ProviderCT", "unit": "µg/m3"},
    "3276405": {"area": "An Giang", "location_name": "Việt Nam", "latitude": 10.5216, "longitude": 105.1259, "owner_name": "Hoaze", "provider": "ProviderAG", "unit": "µg/m3"},
    "3276406": {"area": "Cà Mau", "location_name": "Việt Nam", "latitude": 9.1769, "longitude": 105.1524, "owner_name": "Hoaze", "provider": "ProviderCM", "unit": "µg/m3"}
}

sensor_ids = list(additional_sensors.keys())
sensor_info = additional_sensors

# --- Hàm sinh dữ liệu 1 sensor 1 giờ ---
def generate_sensor_data(sensor_id, dt):
    meta = sensor_info[sensor_id]

    # --- xác định loại khu vực ---
    urban_areas = ["Hà Nội","TP. Hồ Chí Minh","Hải Phòng","Đà Nẵng","Cần Thơ"]
    is_urban = meta['area'] in urban_areas

    # --- giá trị cơ bản theo loại khu vực ---
    base_pm25 = 30 if is_urban else 15
    base_pm10 = 40 if is_urban else 20
    base_no2 = 20 if is_urban else 10
    base_so2 = 10 if is_urban else 5
    base_co = 1.2 if is_urban else 0.8
    base_o3 = 12 if is_urban else 8

    # --- hiệu ứng giờ ---
    if 6 <= dt.hour <= 9 or 17 <= dt.hour <= 20:
        hour_factor = 1.5  # giờ cao điểm giao thông
    elif 0 <= dt.hour <= 5:
        hour_factor = 0.8  # đêm khuya
    else:
        hour_factor = 1.0

    # --- ngày lễ hoặc cuối tuần ---
    holiday_factor = 0.9 if is_holiday(dt) else 1.0  # cuối tuần giảm giao thông

    # --- mùa ---
    if meta['area'] in ["Hà Nội","Hải Phòng","Thái Nguyên","Bắc Ninh"]:
        # miền Bắc
        month_factor = 1.3 if dt.month in [11,12,1,2] else 1.0
    else:
        month_factor = 1.0

    # --- sinh ngẫu nhiên với noise nhỏ ---
    pm25 = base_pm25 * hour_factor * holiday_factor * month_factor * np.random.uniform(0.9,1.1)
    pm10 = base_pm10 * hour_factor * holiday_factor * month_factor * np.random.uniform(0.9,1.1)
    no2 = base_no2 * hour_factor * holiday_factor * np.random.uniform(0.9,1.1)
    so2 = base_so2 * hour_factor * holiday_factor * np.random.uniform(0.9,1.1)
    co = base_co * hour_factor * holiday_factor * np.random.uniform(0.9,1.1)
    o3 = base_o3 * hour_factor * holiday_factor * np.random.uniform(0.9,1.1)

    # --- tính AQI ---
    aqi_pm25 = pm25
    aqi_pm10 = pm10
    aqi_no2 = no2 * 2
    aqi_so2 = so2 * 2
    aqi_co = co * 1.5
    aqi_o3 = o3 * 1.2
    aqi_total = max(aqi_pm25, aqi_pm10, aqi_no2, aqi_so2, aqi_co, aqi_o3)

    pollutants = ['co','no2','so2','pm25','pm10','o3']
    aqi_values = [aqi_co, aqi_no2, aqi_so2, aqi_pm25, aqi_pm10, aqi_o3]
    main_pollutant = pollutants[np.argmax(aqi_values)]

    return {
        'sensor_id': int(sensor_id),
        'area': meta['area'],
        'location_name': meta.get('location_name', meta['area']),
        'datetimeLocal': dt,
        'timezone': 'Asia/Ho_Chi_Minh',
        'latitude': meta['latitude'],
        'longitude': meta['longitude'],
        'owner_name': meta.get('owner_name','Unknown'),
        'provider': meta.get('provider','Unknown'),
        'co_avg': round(co, 2),
        'no2_avg': round(no2, 2),
        'so2_avg': round(so2, 2),
        'pm25_avg': round(pm25, 2),
        'pm10_avg': round(pm10, 2),
        'o3_avg': round(o3, 2),
        'o3_8h_avg': round(o3, 2),
        'aqi_co': round(aqi_co, 2),
        'aqi_no2': round(aqi_no2, 2),
        'aqi_so2': round(aqi_so2, 2),
        'aqi_pm25': round(aqi_pm25, 2),
        'aqi_pm10': round(aqi_pm10, 2),
        'aqi_o3': round(aqi_o3, 2),
        'aqi_total': round(aqi_total, 2),
        'main_pollutant': main_pollutant,
        'unit': meta['unit'],
        'inserted_at': datetime.now()
    }

# --- sinh dữ liệu cả năm theo batch ---
start_date = datetime(2020,1,1)
end_date = datetime(2026,1,31)
hours_per_day = 24
batch_size = 5000  # insert batch tránh full RAM

client = clickhouse_connect.get_client(
    host='localhost',
    user='default',
    password='admin',
    database='hoaze',
    secure=False
)

current_date = start_date
batch_data = []

while current_date <= end_date:
    for h in range(hours_per_day):
        dt = current_date + timedelta(hours=h)
        for sensor_id in sensor_ids:
            batch_data.append(generate_sensor_data(sensor_id, dt))

            # --- insert theo batch ---
            if len(batch_data) >= batch_size:
                df_batch = pd.DataFrame(batch_data)
                client.insert_df('air_quality_analytics', df_batch)
                print(f"Inserted {len(batch_data)} records up to {dt}")
                batch_data = []

    current_date += timedelta(days=1)

# --- insert phần còn lại ---
if batch_data:
    df_batch = pd.DataFrame(batch_data)
    client.insert_df('air_quality_analytics', df_batch)
    print(f"Inserted remaining {len(batch_data)} records")

print("✅ Hoàn tất sinh và chèn dữ liệu vào ClickHouse")

Inserted 5000 records up to 2020-01-07 03:00:00
Inserted 5000 records up to 2020-01-13 06:00:00
Inserted 5000 records up to 2020-01-19 09:00:00
Inserted 5000 records up to 2020-01-25 12:00:00
Inserted 5000 records up to 2020-01-31 15:00:00
Inserted 5000 records up to 2020-02-06 18:00:00
Inserted 5000 records up to 2020-02-12 21:00:00
Inserted 5000 records up to 2020-02-19 00:00:00
Inserted 5000 records up to 2020-02-25 03:00:00
Inserted 5000 records up to 2020-03-02 06:00:00
Inserted 5000 records up to 2020-03-08 09:00:00
Inserted 5000 records up to 2020-03-14 12:00:00
Inserted 5000 records up to 2020-03-20 15:00:00
Inserted 5000 records up to 2020-03-26 18:00:00
Inserted 5000 records up to 2020-04-01 21:00:00
Inserted 5000 records up to 2020-04-08 00:00:00
Inserted 5000 records up to 2020-04-14 03:00:00
Inserted 5000 records up to 2020-04-20 07:00:00
Inserted 5000 records up to 2020-04-26 10:00:00
Inserted 5000 records up to 2020-05-02 13:00:00
Inserted 5000 records up to 2020-05-08 1